# 前臂 EMG：固定格式与可复现分析示例

本 notebook 随 `data/` 文件夹即可运行，无需完整项目、Qt 或设备 SDK。代码与注释以中文说明为主，规范标签保持英文。

**固定契约：** `schema_version=3`，`EMG_FOREARM_CAPTURE_V1`，类别顺序为 `rest/fist/wrist_flexion/wrist_extension`。请先阅读同目录 `README.md`。

流程：校验发布清单 → 读取原始信号/标签 → 审查来源、条件、通道 → 连续因果滤波 → 逐动作特征 → 先分组再切窗 → 仅训练集拟合标准化 → 保存参数与输入哈希。

依赖：NumPy、SciPy；Matplotlib 为可选绘图依赖。编辑器的 Jupyter/Python 内核需另行准备。本示例不训练模型，不修改归档，也不把分类标签解释为连续机器人速度。

In [ ]:
# 1. 基础依赖：数据读取器在本 notebook 内，无项目包导入。
from pathlib import Path
import csv
import hashlib
import html
import io
import json
import math
import os
import struct
import sys

import numpy as np
import scipy
from scipy.signal import butter, iirnotch, sosfilt, tf2sos, welch

try:
    import matplotlib
    import matplotlib.pyplot as plt
except ImportError:
    matplotlib = None
    plt = None
    print("未安装 Matplotlib：绘图单元会跳过，读取、统计与划分示例仍可运行。")

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None


def show_table(rows, limit=20):
    # 在 Jupyter 显示表格；普通 Python 执行时打印 JSON，便于离线核对。
    if not rows:
        print("（无记录）")
        return
    visible = rows[:limit]
    if HTML is None:
        print(json.dumps(visible, ensure_ascii=False, indent=2))
        return
    columns = list(visible[0])
    header = "".join(f"<th>{html.escape(k)}</th>" for k in columns)
    body = "".join("<tr>" + "".join(f"<td>{html.escape(str(r[k]))}</td>" for k in columns) + "</tr>" for r in visible)
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

print("Python:", sys.version.split()[0], "NumPy:", np.__version__, "SciPy:", scipy.__version__)


In [ ]:
# 2. 配置：可填写绝对路径；默认支持从 data/ 或项目根目录启动。
DATA_ROOT_OVERRIDE = ""
USE_MANIFEST = True                 # 已发布的数据建议始终开启。
SELECT_RECORD_ID = None              # None 表示选择清单中第一份已验证记录。
WINDOW_SECONDS = 0.5
STEP_SECONDS = 0.1
SPLIT_SEED = 20261115
WRITE_OUTPUTS = True


def locate_data_root():
    explicit = DATA_ROOT_OVERRIDE or os.environ.get("EMG_EXAMPLE_DATA_ROOT", "")
    if explicit:
        return Path(explicit).expanduser().resolve()
    cwd = Path.cwd()
    for candidate in (cwd, cwd / "data", cwd / "EMG_LAB_2026" / "data"):
        if (candidate / "example_analysis.ipynb").is_file():
            return candidate.resolve()
    raise FileNotFoundError("请在配置单元填写 DATA_ROOT_OVERRIDE，指向收到的 data 文件夹。")


DATA_ROOT = locate_data_root()
OUTPUT_ROOT = Path(os.environ.get("EMG_EXAMPLE_OUTPUT_ROOT", str(DATA_ROOT / "analysis_outputs"))).expanduser().resolve()
# 输出根目录与原始数据根目录分开，原始归档和侧车索引永不覆盖。
if OUTPUT_ROOT == DATA_ROOT:
    raise ValueError("OUTPUT_ROOT 不能等于 DATA_ROOT，请选择独立的分析结果目录。")
print("数据目录：", DATA_ROOT)
print("输出目录：", OUTPUT_ROOT)


## 发布清单与只读基础读取器

下面校验 SHA256、固定版本、二进制长度/通道、标签与计划 schedule 的一致性。时间轴由样本编号重建，原始信号不覆盖。

完整项目读取器还会检查迁移来源链和身份配置；本 notebook 的独立读取器用于已通过发布校验的数据。发布清单提供输入完整性，不是第三方真实性认证。不要根据文件名或目录位置猜测通道含义。

In [ ]:
# 3. 固定格式常量与独立读取器。
SCHEMA_VERSION = 3
PROTOCOL = "EMG_FOREARM_CAPTURE_V1"
ACTIONS = ("rest", "fist", "wrist_flexion", "wrist_extension")
HEADER = struct.Struct("<iiiii10s15sd")
LABEL_FIELDS = ("trial", "start_sample", "end_sample", "class_id", "class_name")


def require(value, message):
    if not value:
        raise ValueError(message)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def read_emg(path):
    # 直接读取 ZIP 成员，不向磁盘解压，不执行归档内容。
    with zipfile_open(path) as archive:
        require(sorted(archive.namelist()) == ["emg.bin", "labels.csv", "meta.json"], "归档成员不正确或存在重复成员。")
        meta = json.loads(archive.read("meta.json").decode("utf-8"))
        binary = archive.read("emg.bin")
        label_text = archive.read("labels.csv").decode("utf-8")
    require(type(meta.get("schema_version")) is int and meta["schema_version"] == SCHEMA_VERSION, "不支持的数据 schema。")
    require(meta.get("task_type") == PROTOCOL and meta.get("capture_only") is True and meta.get("signal_type") == "emg", "不是当前前臂采集协议。")
    require(meta.get("classes") == list(ACTIONS), "类别编号或顺序与 V1 不一致。")
    plan = meta["capture_plan"]
    require(plan.get("protocol") == PROTOCOL and plan.get("origin") in (None, "legacy_custom_v1"), "采集来源不支持。")
    require(meta.get("finalized") is True, "记录尚未完成保存。")
    rate = meta["sampling_rate"]
    require(type(rate) in (int, float) and math.isfinite(rate) and rate > 0, "采样率无效。")
    channels = meta["channel_ids"]
    require(isinstance(channels, list) and channels and all(type(c) is int and c > 0 for c in channels) and len(set(channels)) == len(channels), "通道 ID 无效。")
    require([c["channel_id"] for c in plan["channels"]] == channels, "通道注释与矩阵列顺序不一致。")
    require(plan["movement_side"] in ("left", "right", "both"), "动作侧无效。")
    for channel in plan["channels"]:
        require(channel["side"] in ("left", "right") and channel["position"] in ("forearm_flexor", "forearm_extensor"), "通道侧别/区域无效。")
    require(len(binary) >= HEADER.size, "二进制头不完整。")
    header = HEADER.unpack_from(binary)
    count = meta["samples"]
    require(type(count) is int and count >= 0 and header[0] == HEADER.size and header[1] == len(channels) and header[4] == count, "头信息与元数据不一致。")
    require(len(binary) == HEADER.size + count * (len(channels) + 1) * 4, "二进制字节数不匹配。")
    rows = np.frombuffer(binary, dtype="<f4", offset=HEADER.size).reshape(count, len(channels) + 1)
    require(np.isfinite(rows).all(), "原始信号存在非有限值。")
    reader = csv.DictReader(io.StringIO(label_text))
    require(tuple(reader.fieldnames or ()) == LABEL_FIELDS, "标签列不符合 V1。")
    labels = [{**row, **{k: int(row[k]) for k in LABEL_FIELDS[:-1]}} for row in reader]
    previous_end, seen = 0, set()
    for label in labels:
        a, b, cls = label["start_sample"], label["end_sample"], label["class_id"]
        require(0 <= a < b <= count and a >= previous_end, "标签越界或重叠。")
        require(0 <= cls < len(ACTIONS) and label["class_name"] == ACTIONS[cls], "标签动作编号错误。")
        require(label["trial"] >= 1 and label["trial"] not in seen, "单次动作 ID 重复。")
        previous_end = b
        seen.add(label["trial"])
    schedule = meta["schedule"]
    expected = [{k: row[k] for k in LABEL_FIELDS} for row in schedule if row["end_sample"] <= count]
    require(expected == labels and meta["trial_count"] == len(schedule), "标签与计划 schedule 不一致。")
    require(meta["labels"] == len(labels) and meta["completed_trials"] == len(labels), "完整标签计数不一致。")
    rejected = {r["trial"] for r in meta.get("rejected_trials", [])}
    require(meta["usable_trials"] == sum(r["trial"] not in rejected for r in labels), "有效标签计数不一致。")
    raw_hash = hashlib.sha256(binary).hexdigest()
    if plan.get("origin") == "legacy_custom_v1":
        require(meta.get("migration", {}).get("source_emg_bin_sha256") == raw_hash, "迁移原始二进制哈希失配。")
    # 返回的 raw 是副本；时间列不参与分段，使用样本时钟重建时间。
    return meta, rows[:, 1:].copy(), labels, raw_hash


# 仅依赖 Python 标准库 ZIP 支持。
from zipfile import ZipFile as zipfile_open


In [ ]:
# 4. 根据清单选择并验证输入，默认排除旧原件和模拟测试。
manifest_path = DATA_ROOT / "dataset_manifest.json"
input_hashes = {}
if USE_MANIFEST:
    require(manifest_path.is_file(), "缺少 dataset_manifest.json；请索取完整发布包，或显式关闭清单模式做探索。")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    require(manifest["manifest_version"] == 1 and manifest["format"] == {"schema_version": 3, "task_type": PROTOCOL}, "清单版本/格式不支持。")
    files = []
    for item in manifest["recordings"]:
        path = (DATA_ROOT / item["path"]).resolve()
        require(path.is_relative_to(DATA_ROOT) and path.suffix == ".emg", "清单路径超出数据目录。")
        require(path.is_file() and file_sha256(path) == item["sha256"], f"归档缺失或哈希失配：{path.name}")
        input_hashes[item["path"]] = item["sha256"]
        if item.get("index_path"):
            index_path = (DATA_ROOT / item["index_path"]).resolve()
            require(index_path.is_relative_to(DATA_ROOT) and index_path.is_file(), "发布索引缺失。")
            require(file_sha256(index_path) == item["index_sha256"], "发布索引哈希失配。")
            input_hashes[item["index_path"]] = item["index_sha256"]
        files.append((path, item))
    dataset_id = manifest["dataset_id"]
else:
    # 探索模式不是冻结的数据发布；仍显式跳过旧格式和模拟设备。
    files = []
    for path in sorted(DATA_ROOT.glob("*/recordings/*.emg")):
        with zipfile_open(path) as archive:
            candidate = json.loads(archive.read("meta.json"))
        if candidate.get("task_type") == PROTOCOL and candidate.get("device") != "模拟练习":
            files.append((path, {}))
    dataset_id = "unversioned_exploration"

inventory = []
records = []
seen_ids, seen_raw, seen_paths = set(), set(), set()
for path, declared in files:
    require(path not in seen_paths, "清单重复引用相同文件。")
    seen_paths.add(path)
    meta, signal, labels, raw_hash = read_emg(path)
    require(meta["device"] != "模拟练习", "默认分析包不应包含模拟设备记录。")
    require(meta["record_id"] not in seen_ids and raw_hash not in seen_raw, "重复身份或相同原始数据被重复纳入。")
    if declared:
        require(declared["record_id"] == meta["record_id"] and declared["emg_bin_sha256"] == raw_hash, "清单身份或原始信号哈希失配。")
    seen_ids.add(meta["record_id"])
    seen_raw.add(raw_hash)
    relative = path.relative_to(DATA_ROOT).as_posix()
    input_hashes[relative] = file_sha256(path)
    records.append(dict(path=path, meta=meta, labels=labels))
    inventory.append(dict(受试者=meta["subject"], 文件=path.name, 来源=meta["capture_plan"].get("origin") or "native",
        通道=meta["channel_ids"], 采样率=meta["sampling_rate"], 单位=meta["unit"],
        完整动作=len(labels), 有效动作=meta["usable_trials"], 按计划完成=meta["complete"]))
require(records, "没有符合条件的记录。")
print("数据集：", dataset_id, "记录数：", len(records), "完整动作数：", sum(len(r["labels"]) for r in records))
show_table(inventory)


In [ ]:
# 5. 明确选择一个记录，先避免把不同设备/单位/列顺序直接混合。
selected = next((r for r in records if r["meta"]["record_id"] == SELECT_RECORD_ID), None) if SELECT_RECORD_ID else records[0]
require(selected is not None, "SELECT_RECORD_ID 不在已验证的清单中。")
meta, raw, labels, raw_member_hash = read_emg(selected["path"])
fs = float(meta["sampling_rate"])
time = np.arange(len(raw), dtype=np.float64) / fs
schedule_by_trial = {r["trial"]: r for r in meta["schedule"]}
rejected = {r["trial"] for r in meta.get("rejected_trials", [])}
valid_labels = [r for r in labels if r["trial"] not in rejected]
require(valid_labels, "选择的记录没有完整且有效的动作。")

channel_summary = [dict(矩阵列=i, 通道=c["channel_id"], 侧别=c["side"], 区域=c["position"],
    区域来源=c.get("position_source", "operator_selected_region"), 几何来源=c["placement"]["source"])
    for i, c in enumerate(meta["capture_plan"]["channels"])]
print("选中记录：", selected["path"].name, "record_id：", meta["record_id"])
show_table(channel_summary)
condition_counts = {}
for label in valid_labels:
    s = schedule_by_trial[label["trial"]]
    key = (label["class_name"], s.get("activation_condition", "protocol_default"))
    condition_counts[key] = condition_counts.get(key, 0) + 1
show_table([dict(动作=k[0], 激活条件=k[1], 完整有效次数=n) for k, n in condition_counts.items()])


## 连续因果滤波与描述性分析

在整段记录上滤波一次，保留准备/间隔中的状态连续性，然后按标签切段。滤波器初始状态为零，每个记录重置，不在每个动作开始时重置。示例不做额外动作起止裁剪，也不修改拒绝标记。

绘图只为展示降低采样点密度，统计和特征始终使用完整采样率。SDK raw 幅值不能解释为力或微伏。

In [ ]:
# 6. 与现有实现一致的因果带通 + 陷波；不使用 filtfilt。
FILTER_CONFIG = dict(meta["preprocessing"])


def causal_filter(signal, sampling_rate, config):
    filters = []
    if config["bandpass"]:
        high = min(config["high_hz"], sampling_rate * 0.45)
        require(0 < config["low_hz"] < high, "带通参数超过采样率允许范围。")
        filters.append(butter(4, [config["low_hz"], high], btype="bandpass", fs=sampling_rate, output="sos"))
    if config["notch"]:
        require(0 < config["notch_hz"] < sampling_rate / 2, "陷波频率无效。")
        filters.append(tf2sos(*iirnotch(config["notch_hz"], 30, fs=sampling_rate)))
    if not filters:
        return signal.astype(np.float32, copy=True)
    sos = np.vstack(filters)
    zi = np.zeros((len(sos), 2, signal.shape[1]), dtype=np.float64)
    filtered, _ = sosfilt(sos, signal.astype(np.float64), axis=0, zi=zi)
    return filtered.astype(np.float32)


filtered = causal_filter(raw, fs, FILTER_CONFIG)
trial_features = []
for label in valid_labels:
    planned = schedule_by_trial[label["trial"]]
    segment = filtered[label["start_sample"]:label["end_sample"]].astype(np.float64)
    for col, channel in enumerate(meta["channel_ids"]):
        trial_features.append(dict(record_id=meta["record_id"], subject=meta["subject"], trial=label["trial"],
            round_index=planned["round_index"], action=label["class_name"],
            activation_condition=planned.get("activation_condition", "protocol_default"),
            source_condition_id=planned.get("source_condition_id", ""), channel_id=channel,
            start_sample=label["start_sample"], end_sample=label["end_sample"],
            rms=float(np.sqrt(np.mean(segment[:, col] ** 2))), mav=float(np.mean(np.abs(segment[:, col])))))
show_table(trial_features, limit=12)


In [ ]:
# 7. 可选绘图：原始波形/滤波波形与 Welch 功率谱；结果只写 OUTPUT_ROOT。
if plt is None:
    print("跳过绘图：未安装 Matplotlib。")
else:
    end = min(len(raw), round(30 * fs))
    stride = max(1, end // 10000)
    figure, axes = plt.subplots(len(meta["channel_ids"]), 1, figsize=(12, 3 * len(meta["channel_ids"])), squeeze=False)
    for col, channel in enumerate(meta["channel_ids"]):
        ax = axes[col, 0]
        ax.plot(time[:end:stride], raw[:end:stride, col], label="raw", alpha=0.5)
        ax.plot(time[:end:stride], filtered[:end:stride, col], label="causal filtered", alpha=0.8)
        for label in valid_labels:
            if label["start_sample"] >= end:
                break
            ax.axvspan(label["start_sample"] / fs, min(label["end_sample"], end) / fs, alpha=0.08)
        ax.set(title=f"CH{channel}", xlabel="Time (s)", ylabel=meta["unit"])
        ax.legend()
    figure.tight_layout()
    if WRITE_OUTPUTS:
        OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
        figure.savefig(OUTPUT_ROOT / "waveforms.png", dpi=150)
    plt.show()
    plt.close(figure)
    spectrum_figure, ax = plt.subplots(figsize=(10, 4))
    for col, channel in enumerate(meta["channel_ids"]):
        f, power = welch(filtered[:, col].astype(float), fs=fs, nperseg=min(4096, len(filtered)))
        selected_f = (f > 0) & (f <= min(500, fs / 2))
        ax.semilogy(f[selected_f], np.maximum(power[selected_f], 1e-20), label=f"CH{channel}")
    ax.set(xlabel="Frequency (Hz)", ylabel=f"PSD ({meta['unit']} squared / Hz)", title="Causal filtered signal")
    ax.legend()
    spectrum_figure.tight_layout()
    if WRITE_OUTPUTS:
        spectrum_figure.savefig(OUTPUT_ROOT / "power_spectrum.png", dpi=150)
    plt.show()
    plt.close(spectrum_figure)


## 先划分整轮，再生成滑窗

下面是同一记录内的示例划分。为保留历史整轮结构，使用 `(record_id, round_index)`，同一轮的所有动作以及后续重叠窗口属于同一集合。它只能支持“本次记录内留出轮”的研究，不能当作跨佩戴/跨人泛化。跨记录或跨人研究须在相应更高层级先留出。

只有一个/两个轮时，示例把全部轮分到训练侧并明确不给出验证/测试。这个 notebook 不训练分类器，也不输出准确率。

In [ ]:
# 8. 固定原始组划分；绝不随机打散已经切好的窗口。
group_ids = sorted({int(schedule_by_trial[r["trial"]]["round_index"]) for r in valid_labels})
rng = np.random.default_rng(SPLIT_SEED)
shuffled = rng.permutation(group_ids).tolist()
if len(shuffled) >= 3:
    n_test = max(1, len(shuffled) // 5)
    n_validation = max(1, len(shuffled) // 5)
    split_by_round = {g: "test" if i < n_test else "validation" if i < n_test + n_validation else "train"
                      for i, g in enumerate(shuffled)}
else:
    split_by_round = {g: "train" for g in shuffled}
    print("完整轮不足 3 个：仅演示训练侧准备，不存在独立验证/测试结果。")
split_assignment = [dict(record_id=meta["record_id"], round_index=g,
    split_group=f"{meta['record_id']}:round:{g}", split=split_by_round[g]) for g in group_ids]
show_table(split_assignment)


In [ ]:
# 9. 每个窗口必须完整位于一个已完成且未被拒绝的动作区间内。
window_n = round(WINDOW_SECONDS * fs)
step_n = round(STEP_SECONDS * fs)
require(window_n >= 4 and 1 <= step_n <= window_n, "窗口或步长参数不合法。")
feature_names = [f"ch{channel}_{name}" for channel in meta["channel_ids"] for name in ("rms", "mav")]
window_features = []
feature_vectors = []
for label in valid_labels:
    planned = schedule_by_trial[label["trial"]]
    group = int(planned["round_index"])
    for start in range(label["start_sample"], label["end_sample"] - window_n + 1, step_n):
        end = start + window_n
        segment = filtered[start:end].astype(np.float64)
        values = np.column_stack([np.sqrt(np.mean(segment**2, axis=0)), np.mean(np.abs(segment), axis=0)]).ravel()
        feature_vectors.append(values)
        window_features.append(dict(record_id=meta["record_id"], subject=meta["subject"],
            trial=label["trial"], round_index=group, split_group=f"{meta['record_id']}:round:{group}",
            split=split_by_round[group], class_id=label["class_id"], action=label["class_name"],
            movement_side=meta["capture_plan"]["movement_side"],
            origin=meta["capture_plan"].get("origin") or "native",
            activation_condition=planned.get("activation_condition", "protocol_default"),
            source_condition_id=planned.get("source_condition_id", ""),
            start_sample=start, end_sample=end, unit=meta["unit"],
            **dict(zip(feature_names, values.tolist()))))
require(window_features, "所选参数下没有完整窗口。")
X = np.asarray(feature_vectors, dtype=np.float64)
train_mask = np.array([r["split"] == "train" for r in window_features])
require(train_mask.any(), "没有训练集合窗口。")
# 训练集拟合均值/标准差，验证/测试只应用，零方差特征保持有限。
train_mean = X[train_mask].mean(axis=0)
train_std = X[train_mask].std(axis=0)
train_std = np.where(train_std < 1e-12, 1.0, train_std)
X_normalized = (X - train_mean) / train_std
require(np.isfinite(X_normalized).all(), "标准化产生非有限值。")
for group in group_ids:
    require(len({r["split"] for r in window_features if r["round_index"] == group}) <= 1, "同一原始轮泄漏到多个集合。")
print("特征形状：", X.shape)
show_table([dict(集合=name, 窗口数=sum(r["split"] == name for r in window_features),
    原始轮=[g for g in group_ids if split_by_round[g] == name]) for name in ("train", "validation", "test")])


## 导出可追溯的分析产物

输出保留输入文件/原始信号哈希、单位/通道顺序、来源/激活变体、划分组、样本边界、训练集标准化参数和软件版本。`window_features.csv` 保存标准化前特征，`normalized_features.csv` 保存同一行对应的标准化特征；均不覆盖源数据。

下面最后再次验证发布输入的 SHA256。请不要把 `analysis_outputs/` 当作新的原始数据包。

In [ ]:
# 10. 显式输出目录；所有分析写入都集中在这里。
analysis_parameters = dict(format_contract="forearm-v1.0", schema_version=3, task_type=PROTOCOL,
    dataset_id=dataset_id, input_sha256=input_hashes,
    selected_record_id=meta["record_id"], selected_path=selected["path"].relative_to(DATA_ROOT).as_posix(),
    raw_emg_bin_sha256=raw_member_hash, classes=list(ACTIONS), channel_ids=meta["channel_ids"],
    channel_annotations=meta["capture_plan"]["channels"], unit=meta["unit"], sampling_rate=fs,
    origin=meta["capture_plan"].get("origin") or "native", filtering=FILTER_CONFIG,
    window_seconds=WINDOW_SECONDS, step_seconds=STEP_SECONDS, window_samples=window_n, step_samples=step_n,
    feature_names=feature_names, split_seed=SPLIT_SEED, split_unit="record_id+round_index",
    split_assignment=split_assignment, normalization_fit="train_windows_only",
    train_mean=train_mean.tolist(), train_std=train_std.tolist(),
    software=dict(python=sys.version.split()[0], numpy=np.__version__, scipy=scipy.__version__,
                  matplotlib=matplotlib.__version__ if matplotlib else None),
    model_trained=False, evaluated_accuracy=None)


def save_table(path, rows):
    if rows:
        with path.open("w", encoding="utf-8-sig", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)


if WRITE_OUTPUTS:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    save_table(OUTPUT_ROOT / "trial_features.csv", trial_features)
    save_table(OUTPUT_ROOT / "window_features.csv", window_features)
    normalized_rows = [dict(record_id=r["record_id"], trial=r["trial"], start_sample=r["start_sample"],
        end_sample=r["end_sample"], split=r["split"], **dict(zip(feature_names, z.tolist())))
        for r, z in zip(window_features, X_normalized)]
    save_table(OUTPUT_ROOT / "normalized_features.csv", normalized_rows)
    save_table(OUTPUT_ROOT / "split_assignment.csv", split_assignment)
    (OUTPUT_ROOT / "analysis_parameters.json").write_text(json.dumps(analysis_parameters, ensure_ascii=False,
        indent=2, allow_nan=False), encoding="utf-8")
for relative, expected_hash in input_hashes.items():
    require(file_sha256(DATA_ROOT / relative) == expected_hash, f"分析改变了输入：{relative}")
print("已验证输入文件未改变。输出：", OUTPUT_ROOT if WRITE_OUTPUTS else "未写出（WRITE_OUTPUTS=False）")


## 接下来如何扩展

- 先确定研究问题：动作类别、普通/加强激活区别、跨佩戴或跨受试者泛化。不要把这些不同目标的结果混在一起。
- 若合并记录，先校验设备/单位/采样率、通道语义和动作侧；使用清单、原始哈希和原轮分组排除重复数据/数据泄漏。
- 附加特征/网络或降维模型只能使用训练集合拟合。验证/测试保持隔离，并随实验保存代码版本与配置。
- 本示例不会解除当前项目 `capture_only` 训练门禁，也不生成机器人速度回归标签。